# Runablepassthrough , RunableLambda , RunableParallel

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAI
from langchain_ollama import ChatOllama

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    max_tokens=30
)


Callable function in lambda , class with __call__ can be executed with ()

In [7]:
def add_one(x):
    return x + 1
add_two = lambda x:x+2
print(add_one.__call__(1))
print(add_two.__call__(1))

2
3


Runnable - wrapping callable

In [10]:
from langchain_core.runnables import RunnableLambda
runnable = RunnableLambda(add_two)

print(runnable.invoke(1))

3


1.RunnablePassthrough - just call that only like a pass fun in python 

In [12]:
from langchain_core.runnables import RunnablePassthrough
runpassthrough = RunnablePassthrough()

print(runpassthrough.invoke("hi I am yasin"))

hi I am yasin


RunnablePassthrough.assign - we gives function for this using **.assign**

In [19]:
runpassthrough = RunnablePassthrough.assign(
    lenght = lambda x : len(x['text']),
    upper = lambda x : x['text'].upper()
)
result = runpassthrough.invoke({'text':"yasin mass"})
print(result)

{'text': 'yasin mass', 'lenght': 10, 'upper': 'YASIN MASS'}


2. RunnableLambda — Wrap any function
- What: Turns a plain Python function into a LangChain component
- Why: So your custom function can work inside a LangChain chain using |
- When to use: When you want to add your own logic inside a chain

In [20]:
from langchain_core.runnables import RunnableLambda

# Simple function
def make_uppercase(text):
    return text.upper()

# Turn it into a Runnable
uppercase_step = RunnableLambda(make_uppercase)

result = uppercase_step.invoke("hello world")
print(result)

HELLO WORLD


In [26]:
def add_symbol(text):
    return text + "(*^*)"

symbol = (RunnableLambda(add_symbol)|
          RunnableLambda(make_uppercase)
          )
print(symbol.invoke("yasin"))    #in the output my name is on uppercase and then the symbol is added 

YASIN(*^*)


3. RunnableParallel - run multiple runnables at the same time and merge the result

- What: Runs multiple runnables simultaneously on the same input
- Why: Faster! Instead of running one by one, runs all at once
- When to use: When you need multiple outputs from the same input

In [33]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

def fun1(text):
    return text + "1"
def fun2(text):
    return text + "2"
def fun3(text):
    return text + "3"

chain = RunnableParallel(
    fun1 = RunnableLambda(fun1),
    fun2=RunnableLambda(fun2) ,
    fun3= RunnableLambda(fun3)
)

result = chain.invoke("yasin")
print(result)

{'fun1': 'yasin1', 'fun2': 'yasin2', 'fun3': 'yasin3'}



Merging all - RunnablePassthrough, RunnableLambda, RunnableParallel in single code

In [31]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

# Simple functions
def count_words(text):
    return len(text.split())

def get_first_word(text):
    return text.split()[0]

# Build a simple chain
chain = (
    RunnablePassthrough.assign(
        analysis=RunnableParallel(
            word_count=RunnableLambda(lambda d: count_words(d["text"])),
            first_word=RunnableLambda(lambda d: get_first_word(d["text"]))
        )
    )
)

# Test it
result = chain.invoke({"text": "Python programming is fun"})
print(result)

{'text': 'Python programming is fun', 'analysis': {'word_count': 4, 'first_word': 'Python'}}
